# CELIAC-SAFE RESTAURANT FINDER

This notebook prepares restaurant review data, detects gluten-related signals, classifies review safety, builds a restaurant ranking, and exports the final datasets used by the Streamlit app.

## Workflow

- Define the problem statement
- Load and clean restaurant reviews
- Normalize review text
- Detect gluten-related mentions
- Classify reviews as safe or unsafe
- Aggregate results by restaurant
- Add geolocation data
- Export processed datasets

## 1) Defining the problem statement

The goal of this project is to identify restaurants that may be safer for people with celiac disease using customer reviews.

The notebook analyzes review text, detects gluten-related signals, estimates restaurant safety, and creates a ranked dataset for the interactive Streamlit map.

## 2) Import libraries

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from rapidfuzz import fuzz

## 3) Define project paths

In [ ]:
# Notebook is inside the notebooks/ folder
BASE_DIR = Path("..").resolve()

RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "raw_data.csv"
CLEAN_DATA_PATH = BASE_DIR / "data" / "processed" / "clean_data.csv"
RESTAURANT_RANKING_PATH = BASE_DIR / "data" / "processed" / "restaurant_ranking.csv"

PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 4) Load raw data

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
df.head()

## 5) Data cleaning

Use translated review text when available. If not available, use the original review text.

In [ ]:
df["raw_review_text"] = (
    df["textTranslated"]
    .replace("", pd.NA)
    .combine_first(df["text"])
)

### 5.1 Select relevant columns

In [ ]:
cols = [
    "title",
    "raw_review_text",
    "address",
    "categoryName",
    "price",
    "menu",
    "placeId",
]

for col in cols:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

missing_cols = [col for col in cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

df = df[cols].copy()

### 5.2 Normalize text

In [ ]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[-_/]", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_review_text"] = df["raw_review_text"].apply(normalize_text)

### 5.3 Remove invalid rows and duplicates

In [ ]:
df = df[
    (df["placeId"].notna())
    & (df["placeId"] != "")
    & (df["clean_review_text"] != "")
].copy()

df = df.drop_duplicates(subset=["placeId", "clean_review_text"]).reset_index(drop=True)
df.head()

### 5.4 Save cleaned data

In [ ]:
df.to_csv(CLEAN_DATA_PATH, index=False)
CLEAN_DATA_PATH

## 6) Text feature extraction

Detect gluten-related mentions using regex patterns and fuzzy matching.

In [ ]:
gluten_patterns = [
    r"\bgluten\b",
    r"\bgluten free\b",
    r"\bglutenfree\b",
    r"\bgf\b",
    r"\bceliac\b",
    r"\bcoeliac\b",
    r"\bceliac disease\b",
    r"\bgluten intolerance\b",
    r"\bcross contamination\b",
    r"\bcrosscontamination\b",
    r"\bno cross contamination\b",
    r"\bdedicated fryer\b",
    r"\bseparate fryer\b",
    r"\bdedicated kitchen\b",
    r"\bseparate kitchen\b",
    r"\bgluten safe\b",
    r"\bsafe for celiac\b",
    r"\bceliac friendly\b",
    r"\bgluten friendly\b",
    r"\bgf options\b",
    r"\bgf menu\b",
    r"\bgluten free options\b",
    r"\bgluten free menu\b",
]

combined_pattern = re.compile("|".join(gluten_patterns), flags=re.IGNORECASE)

### 6.1 Detect gluten mentions

In [ ]:
def gluten_match_info(text):
    if not isinstance(text, str) or not text.strip():
        return False, None

    match = combined_pattern.search(text)
    if match:
        return True, match.group(0)

    for word in text.split():
        if len(word) >= 5 and fuzz.ratio(word, "gluten") >= 85:
            return True, f"fuzzy:{word}"

    return False, None


match_result = df["clean_review_text"].apply(gluten_match_info)
df["gluten_match"] = match_result.apply(lambda x: x[0])
df["gluten_trigger"] = match_result.apply(lambda x: x[1])

df[["title", "clean_review_text", "gluten_match", "gluten_trigger"]].head()

## 7) Safety classification

In [ ]:
def has_pattern(text, patterns):
    return any(re.search(pattern, text) for pattern in patterns)


positive_words = [
    "good", "great", "amazing", "excellent", "love", "safe", "delicious", "friendly"
]

negative_words = [
    "bad", "terrible", "awful", "sick", "unsafe", "rude", "horrible"
]


def get_sentiment(text):
    words = text.split()
    positive_count = sum(word in words for word in positive_words)
    negative_count = sum(word in words for word in negative_words)

    return "positive" if positive_count >= negative_count else "negative"

### 7.1 Define safe and unsafe patterns

In [ ]:
safety_patterns = [
    r"gluten[- ]?free",
    r"100 ?% gluten[- ]?free",
    r"separate fryer|dedicated fryer",
    r"dedicated kitchen|dedicated area",
    r"staff (knowledgeable|understood)",
    r"cross[- ]?contamination (aware|awareness|understood)",
    r"celiac safe|coeliac safe",
]

unsafe_patterns = [
    r"got sick",
    r"glutened",
    r"not safe",
    r"unsafe",
    r"cross[- ]?contamination",
]

### 7.2 Classify each review

In [ ]:
def classify_review(text):
    if not isinstance(text, str) or not text.strip():
        return pd.Series({"sentiment": None, "safety": None})

    text = normalize_text(text)
    sentiment = get_sentiment(text)

    if has_pattern(text, unsafe_patterns):
        safety = "unsafe"
    elif has_pattern(text, safety_patterns):
        safety = "safe"
    else:
        safety = None

    return pd.Series({
        "sentiment": sentiment,
        "safety": safety,
    })


df_reviews_classified = pd.concat(
    [
        df.reset_index(drop=True),
        df["clean_review_text"].apply(classify_review),
    ],
    axis=1,
)

df_reviews_classified.head()

## 8) Restaurant safety ranking

In [ ]:
restaurant_ranking = (
    df_reviews_classified
    .groupby("title", as_index=False)
    .agg(
        total_reviews=("clean_review_text", "count"),
        safe_reviews=("safety", lambda x: (x == "safe").sum()),
        unsafe_reviews=("safety", lambda x: (x == "unsafe").sum()),
    )
)

restaurant_ranking["classified_reviews"] = (
    restaurant_ranking["safe_reviews"] + restaurant_ranking["unsafe_reviews"]
)

restaurant_ranking["safety_pct"] = (
    restaurant_ranking["safe_reviews"] / restaurant_ranking["classified_reviews"]
).fillna(0)

restaurant_ranking["ranking_score"] = (
    restaurant_ranking["safe_reviews"] * 2
    - restaurant_ranking["unsafe_reviews"] * 3
    + restaurant_ranking["classified_reviews"] * 0.5
)

restaurant_ranking["safety_label"] = pd.cut(
    restaurant_ranking["safety_pct"],
    bins=[-1, 0.5, 0.7, 1],
    labels=["unsafe", "partly safe", "safe"],
)

### 8.1 Add restaurant metadata

In [ ]:
restaurant_info = (
    df[["title", "address", "placeId", "price", "categoryName", "menu"]]
    .drop_duplicates(subset="title")
)

restaurant_ranking = restaurant_ranking.merge(
    restaurant_info,
    on="title",
    how="left",
)

restaurant_ranking = restaurant_ranking.sort_values(
    by=["ranking_score", "safe_reviews"],
    ascending=False,
).reset_index(drop=True)

restaurant_ranking.head()

## 9) Add geolocation data

This step uses Google Places API to retrieve latitude and longitude from each `placeId`.

**Important:** do not hardcode a real API key in the notebook before uploading to GitHub.

In [ ]:
API_KEY = ""  # Add your Google Places API key locally. Do not commit it to GitHub.


def get_lat_lng(place_id):
    if not API_KEY:
        return None, None

    url = "https://maps.googleapis.com/maps/api/place/details/json"
    params = {
        "place_id": place_id,
        "fields": "geometry",
        "key": API_KEY,
    }

    try:
        response = requests.get(url, params=params, timeout=10).json()
        location = response["result"]["geometry"]["location"]
        return location["lat"], location["lng"]
    except Exception:
        return None, None


restaurant_ranking[["lat", "lng"]] = restaurant_ranking["placeId"].apply(
    lambda place_id: pd.Series(get_lat_lng(place_id))
)

restaurant_ranking.head()

## 10) Export final dataset

In [ ]:
restaurant_ranking.to_csv(RESTAURANT_RANKING_PATH, index=False)
RESTAURANT_RANKING_PATH

## Final note

The exported files are used by the Streamlit app:

- `data/processed/clean_data.csv`
- `data/processed/restaurant_ranking.csv`